# GestuRhythm v2 - Train Conditioned Decoder
Freeze MusicPrior, chi train Cross-Attention weights.
Emotion vector (2D) lam Key/Value, MusicPrior hidden lam Query.
Output: conditioned_decoder.pt

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Load data
sequences  = np.load('/kaggle/input/gesturhythm-v2/sequences.npy')   # (N,30,225)
emotions   = np.load('/kaggle/input/gesturhythm-v2/emotions.npy')    # (N,2)
print(f'sequences: {sequences.shape} | emotions: {emotions.shape}')

In [ ]:
# ── Load pretrained models (frozen) ──
class GestureEmotionEncoder(nn.Module):
    def __init__(self, input_dim=225, d_model=128, nhead=4, num_layers=3):
        super().__init__()
        self.embed   = nn.Linear(input_dim, d_model)
        self.pos_enc = nn.Embedding(512, d_model)
        enc_layer    = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
            dim_feedforward=256, dropout=0.0, batch_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.head = nn.Sequential(nn.Linear(d_model,64),nn.ReLU(),nn.Linear(64,2),nn.Tanh())
    def forward(self, x):
        B,T,_ = x.shape
        mask = torch.triu(torch.ones(T,T,device=x.device),diagonal=1).bool()
        pos  = torch.arange(T,device=x.device).unsqueeze(0)
        x    = self.embed(x) + self.pos_enc(pos)
        return self.head(self.transformer(x,mask=mask)[:,-1,:])

class MusicPrior(nn.Module):
    def __init__(self, vocab_size=130, d_model=128, nhead=4, num_layers=4, max_len=512):
        super().__init__()
        self.embed   = nn.Embedding(vocab_size, d_model, padding_idx=129)
        self.pos_enc = nn.Embedding(max_len, d_model)
        enc_layer    = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
            dim_feedforward=256, dropout=0.0, batch_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, vocab_size)
    def get_hidden(self, tokens):
        B,T = tokens.shape
        mask = torch.triu(torch.ones(T,T,device=tokens.device),diagonal=1).bool()
        pos  = torch.arange(T,device=tokens.device).unsqueeze(0)
        x    = self.embed(tokens) + self.pos_enc(pos)
        return self.transformer(x, mask=mask)  # (B,T,d_model)

enc_ckpt   = torch.load('/kaggle/input/gesturhythm-v2-models/gesture_emotion_encoder.pt', map_location='cpu', weights_only=False)
prior_ckpt = torch.load('/kaggle/input/gesturhythm-v2-models/music_prior.pt',             map_location='cpu', weights_only=False)

encoder = GestureEmotionEncoder(**enc_ckpt['config']).to(device)
encoder.load_state_dict(enc_ckpt['model_state'])
encoder.eval()
for p in encoder.parameters(): p.requires_grad = False

prior = MusicPrior(**prior_ckpt['config']).to(device)
prior.load_state_dict(prior_ckpt['model_state'])
prior.eval()
for p in prior.parameters(): p.requires_grad = False

print('Encoder + MusicPrior loaded and FROZEN')

In [ ]:
# ── Conditioned Decoder: CHI train phan nay ──
class ConditionedDecoder(nn.Module):
    """
    Cross-Attention: emotion_vec lam Key/Value, prior_hidden lam Query.
    Chi co phan nay duoc train, encoder va prior bi freeze.
    """
    def __init__(self, emotion_dim=2, d_model=128, nhead=4, num_layers=2, vocab_size=130):
        super().__init__()
        # Project emotion vector len d_model de dung lam memory
        self.emotion_proj = nn.Linear(emotion_dim, d_model)
        # TransformerDecoder: Query=prior_hidden, Key/Value=emotion_proj
        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=256, dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerDecoder(dec_layer, num_layers=num_layers)
        self.head         = nn.Linear(d_model, vocab_size)

    def forward(self, emotion_vec, prior_hidden):
        """
        emotion_vec:  (B, 2)        -> project -> (B, 1, d_model) = memory
        prior_hidden: (B, T, d_model) = query
        Returns: logits (B, T, vocab_size)
        """
        memory = self.emotion_proj(emotion_vec).unsqueeze(1)  # (B,1,d_model)
        out    = self.transformer(prior_hidden, memory)        # (B,T,d_model)
        return self.head(out)                                  # (B,T,vocab_size)

decoder = ConditionedDecoder().to(device)
total   = sum(p.numel() for p in decoder.parameters())
print(f'ConditionedDecoder params (trainable): {total:,}')

In [ ]:
# ── Dataset: sinh MIDI tokens tu emotion theo mood ──
SCALES = {
    'major':     [60,62,64,65,67,69,71,72,74,76],
    'minor':     [60,62,63,65,67,68,70,72,74,75],
    'pentatonic':[60,62,64,67,69,72,74,76],
}

def emotion_to_token_seq(valence, arousal, seq_len=32):
    """Sinh token sequence phu hop voi cam xuc."""
    if valence > 0.2:
        scale_notes = SCALES['major'] if arousal > 0 else SCALES['pentatonic']
        center      = 67 if arousal > 0.3 else 64
    else:
        scale_notes = SCALES['minor']
        center      = 62 if arousal < -0.2 else 65

    tokens = []
    prev   = center
    for _ in range(seq_len):
        # Chon note gan center, co random walk nho
        candidates = [n for n in scale_notes if abs(n - prev) <= 5]
        if not candidates: candidates = scale_notes
        next_note = candidates[np.random.randint(len(candidates))]
        # Them REST theo arousal thap
        if arousal < -0.3 and np.random.random() < 0.2:
            tokens.append(128)  # REST
        tokens.append(next_note)
        prev = next_note
    return tokens[:seq_len]

class ConditionedDataset(Dataset):
    SEQ_LEN = 32
    def __init__(self, sequences, emotions):
        self.seqs = torch.tensor(sequences, dtype=torch.float32)
        self.emos = torch.tensor(emotions,  dtype=torch.float32)
        # Pre-generate target tokens
        self.targets = []
        for i in range(len(emotions)):
            t = emotion_to_token_seq(float(emotions[i,0]), float(emotions[i,1]), self.SEQ_LEN+1)
            self.targets.append(t)

    def __len__(self): return len(self.seqs)
    def __getitem__(self, i):
        t = self.targets[i]
        return (self.seqs[i], self.emos[i],
                torch.tensor(t[:-1], dtype=torch.long),
                torch.tensor(t[1:],  dtype=torch.long))

dataset  = ConditionedDataset(sequences, emotions)
n_train  = int(0.8 * len(dataset))
train_ds, val_ds = random_split(dataset, [n_train, len(dataset)-n_train],
                                generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32)
print(f'Train: {n_train} | Val: {len(dataset)-n_train}')

In [ ]:
# ── Training: chi update ConditionedDecoder ──
optimizer = torch.optim.Adam(decoder.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
criterion = nn.CrossEntropyLoss(ignore_index=129)

EPOCHS = 50
train_losses, val_losses = [], []

for epoch in range(EPOCHS):
    decoder.train()
    tl = 0
    for gesture_seq, emotion_vec, token_in, token_out in train_loader:
        gesture_seq = gesture_seq.to(device)
        emotion_vec = emotion_vec.to(device)
        token_in    = token_in.to(device)
        token_out   = token_out.to(device)

        # Lay emotion tu encoder (frozen)
        with torch.no_grad():
            pred_emotion = encoder(gesture_seq)          # (B,2)
            prior_hidden = prior.get_hidden(token_in)    # (B,T,128)

        optimizer.zero_grad()
        logits = decoder(pred_emotion, prior_hidden)     # (B,T,130)
        loss   = criterion(logits.reshape(-1,130), token_out.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(decoder.parameters(), 1.0)
        optimizer.step()
        tl += loss.item()

    decoder.eval()
    vl = 0
    with torch.no_grad():
        for gesture_seq, emotion_vec, token_in, token_out in val_loader:
            gesture_seq = gesture_seq.to(device)
            emotion_vec = emotion_vec.to(device)
            token_in    = token_in.to(device)
            token_out   = token_out.to(device)
            pred_emotion = encoder(gesture_seq)
            prior_hidden = prior.get_hidden(token_in)
            logits       = decoder(pred_emotion, prior_hidden)
            vl += criterion(logits.reshape(-1,130), token_out.reshape(-1)).item()

    scheduler.step()
    train_losses.append(tl / len(train_loader))
    val_losses.append(vl / len(val_loader))
    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS} | train={train_losses[-1]:.4f} | val={val_losses[-1]:.4f}')
print('Done!')

In [ ]:
# Bieu do
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Conditioned Decoder Training', fontweight='bold')

axes[0].plot(train_losses, label='Train', color='steelblue')
axes[0].plot(val_losses,   label='Val',   color='coral')
axes[0].set_title('Loss (CrossEntropy)'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

# So sanh output cua 2 cam xuc khac nhau
decoder.eval()
with torch.no_grad():
    happy_emo = torch.tensor([[0.8, 0.8]], dtype=torch.float32).to(device)
    sad_emo   = torch.tensor([[-0.8,-0.8]], dtype=torch.float32).to(device)
    primer    = torch.tensor([[60,64,67]], dtype=torch.long).to(device)

    def gen_with_emotion(emo, n=16):
        tokens = primer.clone()
        for _ in range(n):
            h      = prior.get_hidden(tokens)
            logits = decoder(emo, h)[0, -1, :]
            logits[128:] = float('-inf')  # bo REST/PAD
            next_t = torch.multinomial(F.softmax(logits/0.8, dim=0), 1)
            tokens = torch.cat([tokens, next_t.unsqueeze(0)], dim=1)
        return tokens[0, 3:].cpu().numpy()

    happy_notes = gen_with_emotion(happy_emo)
    sad_notes   = gen_with_emotion(sad_emo)

x = range(len(happy_notes))
axes[1].plot(x, happy_notes, 'o-', color='#2ecc71', label='Happy (+0.8,+0.8)', linewidth=2)
axes[1].plot(x, sad_notes,   's-', color='#e74c3c', label='Sad (-0.8,-0.8)',   linewidth=2)
axes[1].set_title('Generated Melody: Happy vs Sad'); axes[1].set_xlabel('Note index')
axes[1].set_ylabel('MIDI pitch'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('conditioned_decoder_results.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Happy melody avg pitch: {happy_notes.mean():.1f}')
print(f'Sad   melody avg pitch: {sad_notes.mean():.1f}')

In [ ]:
torch.save({
    'model_state': decoder.state_dict(),
    'config': {'emotion_dim':2,'d_model':128,'nhead':4,'num_layers':2,'vocab_size':130},
    'val_loss': val_losses[-1],
}, 'conditioned_decoder.pt')
print('Saved: conditioned_decoder.pt')
print(f'Final val loss: {val_losses[-1]:.4f}')